In [1]:
import os

In [2]:
%pwd

'd:\\Smart-Inventory-Defect-Detection-Quality-Control-API\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\Smart-Inventory-Defect-Detection-Quality-Control-API'

In [34]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainingConfig:
    root_dir: Path
    batch_size: int
    epochs: int
    learning_rate: float
    weight_decay: float
    lr_scheduler: str

In [35]:
from SIDD.constants import *
from SIDD.utils.common import read_yaml, create_directories

In [36]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:
        config = self.config.model_training
        params = self.params.model_training

        model_training_config = ModelTrainingConfig(
            root_dir=Path(config.root_dir),
            batch_size=params.batch_size,
            epochs=params.epochs,
            learning_rate=params.learning_rate,
            weight_decay=params.weight_decay,
            lr_scheduler=params.lr_scheduler
        )

        return model_training_config

In [37]:
import torch
import torch.optim as optim
from SIDD.components.data_preparation import DataPreparation
from SIDD.components.loss_and_metrics import LossAndMetrics
from SIDD.components.model_building import ModelBuilding
from SIDD.components.model_training import ModelTraining

In [38]:
class ModelTraining:
    def __init__(
        self,
        config: ModelTrainingConfig,
        model,
        train_loader,
        val_loader,
        criterion,
    ):   
        
        self.config = config
            
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.model.to(self.device)

        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=self.config.learning_rate,
            weight_decay=self.config.weight_decay
        )

        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer,
            T_max=self.config.epochs
        )


    def train_one_epoch(self):
        self.model.train()

        running_loss = 0.0

        for images, masks in self.train_loader:

            images = images.to(self.device)
            masks = masks.to(self.device)

            self.optimizer.zero_grad()

            outputs = self.model(images)

            loss = self.criterion(outputs, masks)

            loss.backward()

            self.optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(self.train_loader)

        return epoch_loss

    def validate(self):
        self.model.eval()

        running_loss = 0.0

        with torch.no_grad():

            for images, masks in self.val_loader:

                images = images.to(self.device)
                masks = masks.to(self.device)

                outputs = self.model(images)

                loss = self.criterion(outputs, masks)

                running_loss += loss.item()

        epoch_loss = running_loss / len(self.val_loader)

        return epoch_loss

    def train(self):

        best_loss = float("inf")

        for epoch in range(self.config.epochs):

            train_loss = self.train_one_epoch()

            val_loss = self.validate()

            self.scheduler.step()

            print(
                f"Epoch [{epoch+1}/{self.config.epochs}] "
                f"Train Loss: {train_loss:.4f} "
                f"Val Loss: {val_loss:.4f}"
            )

            if val_loss < best_loss:

                best_loss = val_loss

                torch.save(
                    self.model.state_dict(),
                    self.config.root_dir / "best_model.pth"
                )

        print("Training Completed!")
        

In [39]:
config = ConfigurationManager()

# Data
data_config = config.get_data_preparation_config()
data = DataPreparation(data_config)
train_loader, val_loader = data.get_dataloaders()

# Model
model_config = config.get_model_building_config()
model = ModelBuilding(model_config).build_model()

# Loss
loss_config = config.get_loss_metrics_config()
criterion = LossAndMetrics(loss_config).get_bce_dice_loss()

# Trainer
train_config = config.get_model_training_config()

trainer = ModelTraining(
    config=train_config,
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion
)

trainer.train()

[2026-07-19 23:01:45,736: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-19 23:01:45,742: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-19 23:01:45,745: INFO: common: created directory at: artifacts]


AttributeError: 'ConfigurationManager' object has no attribute 'get_data_preparation_config'